# Chronos-Spatial — Stage 1: Frame Extraction (real + ai_generated, combined)

Extracts `N_WINDOWS` **contiguous windows** of `WINDOW_LEN` frames per video,
each **resized to a square** (aspect distorted, no bars, no cropping, no face
detection). Detects fully AI-generated video, whose artifacts are global —
watermarks, morphing backgrounds, texture sliding, and temporal flicker — so
the model must see the WHOLE frame.

**Why windows (not uniform frames)?** The hybrid's ConvLSTM branch needs to see
real MOTION — consecutive frames — not 0.5s-apart snapshots. Each window is a
short contiguous clip; several windows are spread across the video for content
coverage. The EfficientNet spatial branch trains on the individual frames; the
ConvLSTM trains on the window sequences. (`WINDOW_LEN=1` = plain uniform sampling.)

**Two anti-shortcut details baked in here:**
- **Time-normalized stride (`TARGET_FPS`)** — window frames are ~1/12s apart in
  *wall-clock time* for every video. AI generators often render 8–24fps vs real
  cameras 30–60fps; with a fixed source-frame stride the motion step size would
  leak the frame rate, and the ConvLSTM could classify on that instead of
  artifacts.
- **Black-border trim (`TRIM_BLACK_BORDERS`)** — letterbox bars baked into the
  source pixels are stripped (one conservative box per video) before
  resize-square, so bar geometry can't re-encode orientation as a label cue.

**This run does BOTH classes in one pass**, each capped independently
(default 3000 real + 3000 ai_generated), deduplicated globally, with
corrupt and out-of-duration videos skipped and backfilled from the pool.
Output: one `chronos_frames_combined.tar` + `frames_index_combined.csv`
containing both labels — ready for the manifest stage with no merging step.

## Settings for this notebook
- Accelerator: **None (CPU)** — no face detection in v1, so a GPU adds
  nothing here. Save the GPU quota for training.
- Internet: **ON** (for the `decord` install; falls back to OpenCV if off).
- Attach all FIVE datasets via *Add Input*: `aigenerated-3000` and
  `real3000` (the new zipped batches — auto-unzipped below), plus the
  originals `aigvdbench`, `real501`, `aigvdrealvideos`.
  Do NOT attach the old `chronos-diversity-batch` (superseded by the new batches).
- Labels are set by SOURCES below, NOT by folder names: `real501` is misnamed
  (it contains AI video) and is deliberately labeled `ai_generated`.

## Workflow
1. Smoke test: `LIMIT_PER_CLASS = 5`, Run All, check the dedupe report +
   preview grid + per-class summary.
2. Full run: `LIMIT_PER_CLASS = None`, then **Save Version → Save & Run
   All (Commit)**.

In [ ]:
!ls -la /kaggle/input/


In [ ]:
from pathlib import Path

BATCH_TAG = "combined"

def resolve(*candidates: str) -> Path:
    """Returns the first existing path — handles both Kaggle mount forms."""
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    raise FileNotFoundError(f"None of these mounts exist: {candidates}. "
                            "Check the 'Add Input' sidebar and the ls output above.")

import glob as _gg

def find_ds(*names):
    """First attached dataset matching any of these names, wherever mounted."""
    for nm in names:
        for pat in (f"/kaggle/input/{nm}", f"/kaggle/input/*/{nm}",
                    f"/kaggle/input/*/*/{nm}"):
            hits = sorted(_gg.glob(pat))
            if hits:
                return Path(hits[0])
    return None

_WANT = [
    ("real",         ["real-1537"]),
    ("real",         ["realtotal"]),
    ("real",         ["real-portrait2000"]),
    ("real",         ["real-portrait23"]),
    ("real",         ["chronos-real-modern", "real-modern"]),
    ("real",         ["real-gdrive"]),
    ("ai_generated", ["aigenerated-3000"]),
    ("ai_generated", ["ai-portrait-2000"]),
    ("ai_generated", ["aigeneratedtopup-339"]),
    ("ai_generated", ["chronos-ai-diverse", "ai-diverse"]),
    ("ai_generated", ["fake-gdrive"]),
]

SOURCES = []
_seen_src = set()

def _add_source(label, path):
    """Append once; identical paths reached by two routes are collapsed."""
    key = path.resolve()
    if key in _seen_src:
        return
    _seen_src.add(key)
    SOURCES.append((label, path))
    print(f"OK  {label:13s} <- {path}")

for _label, _names in _WANT:
    _d = find_ds(*_names)
    if _d is None:
        print(f"note: {_names[0]} ({_label}) not attached -- skipped (fine if not extracting it this run)")
    else:
        _add_source(_label, _d)


_have = {lbl for lbl, _ in SOURCES}
assert {"real", "ai_generated"} <= _have,     f"a whole class is missing its datasets -- found only {_have}. Check Add Input."

DEDUPE = True

TARGET_PER_CLASS = {"real": 5000, "ai_generated": 5000}
MIN_DURATION     = 5.0
MAX_DURATION     = 12.0

MIN_DURATION_BY_CLASS = {"real": MIN_DURATION, "ai_generated": 3.0}
LIMIT_PER_CLASS  = 5

WINDOW_LEN       = 16
N_WINDOWS        = 2
TARGET_FPS       = 12
FALLBACK_STRIDE  = 2
TRIM_BLACK_BORDERS = True
IMAGE_SIZE       = 380
JPEG_QUALITY     = 90

import time as _time
TIME_BUDGET_HOURS = 10.0
RUN_START = _time.time()
RUN_STAMP = _time.strftime("%m%d_%H%M")

OUT_DIR    = Path("/kaggle/tmp/data")
FRAMES_DIR = OUT_DIR / "frames"
INDEX_CSV  = OUT_DIR / f"frames_index_{BATCH_TAG}_{RUN_STAMP}.csv"

for src, root in SOURCES:
    print(f"{src:14s} <- {root}")

_CONTRACT = {
    "IMAGE_SIZE": (IMAGE_SIZE, 380),
    "WINDOW_LEN": (WINDOW_LEN, 16),
    "N_WINDOWS": (N_WINDOWS, 2),
    "TARGET_FPS": (TARGET_FPS, 12),
    "FALLBACK_STRIDE": (FALLBACK_STRIDE, 2),
    "JPEG_QUALITY": (JPEG_QUALITY, 90),
    "TRIM_BLACK_BORDERS": (TRIM_BLACK_BORDERS, True),
    "MAX_DURATION": (MAX_DURATION, 12.0),
    "BATCH_TAG": (BATCH_TAG, "combined"),
}
print("\nFORMAT CONTRACT vs the previous extraction:")
_drift = []
for _k, (_got, _exp) in _CONTRACT.items():
    _ok = _got == _exp
    print(f"  {_k:20s} {str(_got):10s} expected {str(_exp):10s} {'OK' if _ok else '*** DRIFT'}")
    if not _ok:
        _drift.append(_k)
assert not _drift, (f"FORMAT DRIFT in {_drift} -- these frames would NOT be "
                    f"compatible with hybridframeextraction. Fix before running.")
print("  -> frames will be byte-format identical to the old extraction (mergeable)")
print(f"  note: ai_generated duration floor is {MIN_DURATION_BY_CLASS['ai_generated']}s "
      f"(old run used {MIN_DURATION}s for both classes). This changes WHICH videos "
      f"are accepted, never HOW a frame is written -- format stays identical.")


In [ ]:
!pip -q install decord || echo "decord unavailable -> OpenCV fallback"
import cv2, numpy as np
print("opencv:", cv2.__version__)


In [ ]:
import csv, hashlib, re
import numpy as np

VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".mpg", ".mpeg", ".wmv"}


def stride_for_fps(fps, target_fps, fallback=2):
    """Source-frame stride so window frames are ~1/target_fps sec apart --
    time-normalized motion, so the ConvLSTM can't read the source frame rate
    off the motion step size."""
    if not fps or fps <= 0 or not target_fps or target_fps <= 0:
        return max(1, int(fallback))
    return max(1, int(round(fps / target_fps)))


def black_border_box(rgb, thresh=12, bar_frac=0.99, max_trim_frac=0.40):
    """Content box (top, bottom, left, right; exclusive ends) after stripping
    near-black border rows/cols. Conservative: falls back to the full frame
    when the result would be degenerate (e.g. a black fade frame)."""
    h, w = rgb.shape[:2]
    dark = rgb.max(axis=2) < thresh
    row_bar = dark.mean(axis=1) >= bar_frac
    col_bar = dark.mean(axis=0) >= bar_frac
    max_v, max_h = int(h * max_trim_frac), int(w * max_trim_frac)
    top = 0
    while top < max_v and row_bar[top]:
        top += 1
    bottom = h
    while bottom > h - max_v and row_bar[bottom - 1]:
        bottom -= 1
    left = 0
    while left < max_h and col_bar[left]:
        left += 1
    right = w
    while right > w - max_h and col_bar[right - 1]:
        right -= 1
    if bottom - top < h * 0.3 or right - left < w * 0.3:
        return 0, h, 0, w
    return top, bottom, left, right


def common_content_box(frames):
    """MINIMAL trim across frames: remove a border only if it's a bar in EVERY
    frame. Robust to fades, and gives all of a video's frames one geometry."""
    h, w = frames[0].shape[:2]
    boxes = [black_border_box(f) for f in frames]
    top, bottom = min(b[0] for b in boxes), max(b[1] for b in boxes)
    left, right = min(b[2] for b in boxes), max(b[3] for b in boxes)
    if bottom - top < h * 0.3 or right - left < w * 0.3:
        return 0, h, 0, w
    return top, bottom, left, right


def plan_windows(total, window_len, n_windows, frame_stride):
    """Up to n_windows lists of window_len CONTIGUOUS source-frame indices,
    spread across [0, total). Shrinks stride to fit short clips; clamps (pads)
    the tail for clips shorter than one window. window_len=1 -> uniform."""
    if total <= 0 or window_len < 1 or n_windows < 1:
        return []
    stride = max(1, int(frame_stride))
    while stride > 1 and (window_len - 1) * stride + 1 > total:
        stride -= 1
    span = (window_len - 1) * stride + 1
    last_start = max(0, total - span)
    if n_windows == 1:
        starts = [last_start // 2]
    else:
        starts = [int(round(k * last_start / (n_windows - 1))) for k in range(n_windows)]
    starts = sorted(dict.fromkeys(starts))
    return [[min(s + i * stride, total - 1) for i in range(window_len)] for s in starts]


class DurationSkip(Exception):
    """Video is fine but its length is outside [MIN_DURATION, MAX_DURATION]."""


def _read_windows_opencv(path, window_len, n_windows, target_fps, fallback_stride, min_dur, max_dur):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError("opencv cannot open")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur = total / fps if fps else 0.0
    if dur and not (min_dur <= dur <= max_dur):
        cap.release()
        raise DurationSkip(f"{dur:.1f}s")
    plans = plan_windows(total, window_len, n_windows, stride_for_fps(fps, target_fps, fallback_stride))
    if not plans:
        cap.release()
        raise ValueError("opencv: 0 frames")
    flat = sorted({i for w in plans for i in w})
    lut = {}
    for i in flat:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, bgr = cap.read()
        if ok:
            lut[i] = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    cap.release()
    windows = [[(i, i / fps, lut[i]) for i in w] for w in plans if all(i in lut for i in w)]
    if not windows:
        raise ValueError("opencv decoded no complete window")
    return windows, dur


def read_windows(path, window_len, n_windows, target_fps, fallback_stride, min_dur, max_dur):
    """Returns (windows, duration_sec) where windows is a list of
    [(frame_index, time_sec, rgb), ...], each of length window_len and
    temporally CONTIGUOUS at ~target_fps wall-clock steps -> the ConvLSTM sees
    real motion at the SAME time scale for every source frame rate.
    n_windows windows are spread across the clip.

    Robust by design:
      * Probes duration first; raises DurationSkip if out of range (cheap).
      * decord, then FALLS BACK to OpenCV on ANY decord failure.
      * Truly undecodable files raise ValueError -> caller skips as corrupt.
    """
    try:
        import decord
    except ImportError:
        return _read_windows_opencv(path, window_len, n_windows, target_fps, fallback_stride, min_dur, max_dur)

    try:
        reader = decord.VideoReader(str(path))
        fps = float(reader.get_avg_fps()) or 30.0
        total = len(reader)
        dur = total / fps if fps else 0.0
        if dur and not (min_dur <= dur <= max_dur):
            raise DurationSkip(f"{dur:.1f}s")
        plans = plan_windows(total, window_len, n_windows, stride_for_fps(fps, target_fps, fallback_stride))
        if not plans:
            raise ValueError("decord: 0 frames")
        flat = sorted({i for w in plans for i in w})
        batch = reader.get_batch(flat).asnumpy()
        lut = {i: f for i, f in zip(flat, batch)}
        return [[(i, i / fps, lut[i]) for i in w] for w in plans], dur
    except DurationSkip:
        raise
    except Exception:
        return _read_windows_opencv(path, window_len, n_windows, target_fps, fallback_stride, min_dur, max_dur)


def resize_square(rgb: np.ndarray, size: int) -> np.ndarray:
    """Resize the WHOLE frame to a size x size square, distorting the aspect
    ratio (no padding, no cropping). Keeps every pixel. Letterbox padding is
    avoided on purpose: black bars would encode orientation, which the model
    can latch onto as a spurious real/fake cue (a portrait clip would become
    mostly black -> read as fake). Squashing keeps orientation OUT of the
    signal -- portrait and landscape become the same shape. Inference resizes
    identically."""
    return cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)


def fingerprint(path: Path) -> str:
    """Cheap content identity: file size + first 1MB. Identical files copied
    between datasets always match; different videos essentially never do."""
    hsh = hashlib.sha1(str(path.stat().st_size).encode())
    with open(path, "rb") as f:
        hsh.update(f.read(1024 * 1024))
    return hsh.hexdigest()


def make_video_id(source: str, video_path: Path, root: Path) -> str:
    rel = f"{source}/{root.name}/{video_path.relative_to(root).as_posix()}"
    digest = hashlib.sha1(rel.encode()).hexdigest()[:8]
    stem = re.sub(r"[^A-Za-z0-9_-]", "_", video_path.stem)[:80]
    return f"{source}__{stem}__{digest}"

print("library ready (full-frame resize-to-square mode -- no face detection)")


In [ ]:
import zipfile
from collections import Counter as _Cnt

UNPACK_DIR = Path("/kaggle/tmp/unpacked")
resolved_sources = []
for label, root in SOURCES:
    resolved_sources.append((label, root))
    zips = sorted(root.rglob("*.zip")) if root.is_dir() else []
    if not zips:
        continue
    dest = UNPACK_DIR / root.name
    if not dest.exists():
        dest.mkdir(parents=True, exist_ok=True)
        for z in zips:
            print(f"unzipping {z.relative_to(root).as_posix()} "
                  f"({z.stat().st_size / 1e9:.2f} GB) -> {dest}")
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(dest)
            except Exception as e:
                print(f"  !! unreadable zip {z.name}: {e}")
    n = sum(1 for _ in dest.rglob("*.mp4"))
    print(f"{label}: {root.name} had {len(zips)} zip(s) -> {dest} ({n} videos)")
    resolved_sources.append((label, dest))
SOURCES = resolved_sources

print("\nINVENTORY per source (nested folders shown):")
for label, root in SOURCES:
    if not root.exists():
        print(f"  {label:13s} {root}   *** PATH MISSING")
        continue
    mp4s = list(root.rglob("*.mp4"))
    nzip = sum(1 for _ in root.rglob("*.zip"))
    print(f"  {label:13s} {root.name:26s} mp4={len(mp4s):5d}  zip={nzip}")
    for sub, k in sorted(_Cnt(p.parent.relative_to(root).as_posix()
                              for p in mp4s).items()):
        print(f"        - {(sub or '(root)'):44s} {k:5d}")


In [ ]:
import glob as _glob
import pandas as _pd

PREV_DONE = set()
PREV_COUNTS = {}
for _csv in _glob.glob("/kaggle/input/**/frames_index_*.csv", recursive=True):
    try:
        df_prev = _pd.read_csv(_csv, usecols=["video_id", "source"]).drop_duplicates("video_id")
        PREV_DONE |= set(df_prev["video_id"])
        for src, n in df_prev["source"].value_counts().items():
            PREV_COUNTS[src] = PREV_COUNTS.get(src, 0) + int(n)
        print(f"resume: {_csv} -> {len(df_prev)} videos already extracted")
    except Exception as e:
        print(f"resume: skipping unreadable {_csv}: {e}")
print(f"total previously-extracted videos to skip: {len(PREV_DONE)} {PREV_COUNTS or ''}")


In [ ]:
from tqdm.auto import tqdm
from collections import Counter

_REAL_PREFIXES = ("ugc_", "vis_", "pexl_", "pex_")
_AI_PREFIXES = ("gvb_", "avg_", "dact_")


def prefix_class(stem):
    """Class implied by the filename prefix, or None when unrecognised
    (older pools have arbitrary names -> trust the folder label for those)."""
    if stem.startswith(_AI_PREFIXES):
        return "ai_generated"
    if stem.startswith(_REAL_PREFIXES):
        return "real"
    return None


_OPPOSITE = {"real": "ai_generated", "ai_generated": "real"}

candidates = []
census = {}
n_mislabelled = 0
n_crossclass = 0
for source, root in SOURCES:
    vids = sorted(p for p in root.rglob("*") if p.suffix.lower() in VIDEO_EXTS and p.is_file())
    _opp = _OPPOSITE[source]
    _cross = [p for p in vids if _opp in p.parts]
    if _cross:
        n_crossclass += len(_cross)
        vids = [p for p in vids if _opp not in p.parts]
    good = [p for p in vids if p.stat().st_size >= 50_000]
    n_tiny = len(vids) - len(good)
    _clash = [p for p in good
              if (pc := prefix_class(p.stem)) is not None and pc != source]
    if _clash:
        n_mislabelled += len(_clash)
        print(f"  *** {len(_clash)} file(s) under '{source}' have a CONTRADICTING "
              f"prefix -> DROPPED (e.g. {_clash[0].name})")
        _bad = {p for p in _clash}
        good = [p for p in good if p not in _bad]
    gb = sum(p.stat().st_size for p in good) / 1e9
    candidates += [(source, root, p) for p in good]
    census[(source, str(root))] = (len(good), n_tiny, gb)
    _disp = root.name if root.name not in ("real", "ai_generated")         else f"{root.parent.name}/{root.name}"
    print(f"{source:13s} <- {_disp:34s} {len(good):6d} videos {gb:8.2f} GB"
          + (f"   ({n_tiny} tiny/suspect EXCLUDED)" if n_tiny else "")
          + (f"   [{len(_cross)} cross-class EXCLUDED]" if _cross else ""))

print("\nlabel-safety: %d file(s) dropped for a prefix that contradicts their "
      "source label%s" % (n_mislabelled, " -- INVESTIGATE" if n_mislabelled else " (clean)"))
print("cross-class : %d file(s) skipped for sitting under the opposite class "
      "folder (expected when a dataset carries both)" % n_crossclass)

print("\n=== TOTAL AVAILABLE, per class (before duration/corruption filtering) ===")
for label in ("real", "ai_generated"):
    n = sum(v[0] for (s, _), v in census.items() if s == label)
    tgt = TARGET_PER_CLASS.get(label, 0)
    print(f"  {label:13s} {n:6d} available   target {tgt}   "
          + ("OK" if n >= tgt else f"SHORT by {tgt - n}"))
print(f"  {'GRAND TOTAL':13s} {sum(v[0] for v in census.values()):6d} videos")

if DEDUPE:
    seen, unique, dups = {}, [], 0
    for source, root, p in tqdm(candidates, desc="dedupe"):
        fp = fingerprint(p)
        if fp in seen:
            dups += 1
            continue
        seen[fp] = p
        unique.append((source, root, p))
    print(f"dedupe: {len(candidates)} candidates -> {len(unique)} unique ({dups} duplicates dropped)")
    candidates = unique

pool_counts = Counter(s for s, _, _ in candidates)
print("\npool available for extraction (duration range per class: " +
      ", ".join(f"{k}={MIN_DURATION_BY_CLASS.get(k, MIN_DURATION):.1f}-{MAX_DURATION:.0f}s"
                for k in TARGET_PER_CLASS) + "):")
for label, target in TARGET_PER_CLASS.items():
    have = pool_counts.get(label, 0)
    flag = "OK" if have >= target else f"SHORT by {target - have} (before duration/corruption filtering)"
    print(f"  {label:14s} pool={have:5d}  target={target:5d}  {flag}")


In [ ]:
import shutil

FRAMES_DIR.mkdir(parents=True, exist_ok=True)


def extract_one(video_path, video_id, out_dir, source):
    """Decode CONTIGUOUS windows + bar-trim + resize-square + write one video.
    Returns row dicts, or raises DurationSkip / Exception (caller skips). A
    window is written only if COMPLETE + clean (fixed length for the ConvLSTM)."""
    windows, _dur = read_windows(str(video_path), WINDOW_LEN, N_WINDOWS,
                                 TARGET_FPS, FALLBACK_STRIDE,
                                 MIN_DURATION_BY_CLASS.get(source, MIN_DURATION),
                                 MAX_DURATION)
    box = None
    if TRIM_BLACK_BORDERS:
        all_rgb = [rgb for window in windows for _, _, rgb in window
                   if rgb is not None and getattr(rgb, "size", 0) > 0]
        if all_rgb:
            box = common_content_box(all_rgb)
    out_dir.mkdir(parents=True, exist_ok=True)
    local_rows = []
    for w_idx, window in enumerate(windows):
        if any(rgb is None or getattr(rgb, "size", 0) == 0 or rgb.shape[0] < 2 or rgb.shape[1] < 2
               for _, _, rgb in window):
            continue
        for pos, (fidx, t, rgb) in enumerate(window):
            if box is not None:
                bt, bb, bl, br = box
                rgb = rgb[bt:bb, bl:br]
            oh, ow = int(rgb.shape[0]), int(rgb.shape[1])
            img = resize_square(rgb, IMAGE_SIZE)
            name = f"w{w_idx:02d}_p{pos:02d}_f{fidx:06d}.jpg"
            cv2.imwrite(str(out_dir / name), cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            local_rows.append(dict(video_id=video_id, video_path=str(video_path),
                                   frame_path=(out_dir / name).relative_to(OUT_DIR).as_posix(),
                                   frame_index=fidx, time_sec=round(t, 3),
                                   source=source, stem=video_path.stem, orig_w=ow, orig_h=oh,
                                   window_index=w_idx, pos_in_window=pos))
    if not local_rows:
        raise ValueError("no valid window written")
    return local_rows


targets = dict(TARGET_PER_CLASS)
if LIMIT_PER_CLASS is not None:
    targets = {k: min(v, LIMIT_PER_CLASS) for k, v in targets.items()}
    print(f"SMOKE TEST: capping each class at {LIMIT_PER_CLASS} -- set LIMIT_PER_CLASS = None for the full run")
for _src, _n in PREV_COUNTS.items():
    if _src in targets:
        targets[_src] = max(0, targets[_src] - _n)
        print(f"resume: {_src} target reduced by {_n} already-extracted -> {targets[_src]} this run")

rows = []
n_ok = {k: 0 for k in targets}
n_corrupt = {k: 0 for k in targets}
n_duration = {k: 0 for k in targets}
n_by_root = Counter()
n_prev_skipped = 0
n_logged = 0
budget_hit = False

pbar = tqdm(candidates, desc="extract")
for source, root, video_path in pbar:
    if source not in targets:
        continue
    if n_ok[source] >= targets[source]:
        continue

    if _time.time() - RUN_START > TIME_BUDGET_HOURS * 3600:
        budget_hit = True
        print(f"\n*** TIME BUDGET ({TIME_BUDGET_HOURS}h) reached -- stopping cleanly to package output.")
        break

    video_id = make_video_id(source, video_path, root)
    if video_id in PREV_DONE:
        n_prev_skipped += 1
        continue
    out_dir = FRAMES_DIR / source / video_id

    existing = sorted(out_dir.glob("w*.jpg")) if out_dir.exists() else []
    if existing:
        for fp in existing:
            m = re.match(r"w(\d+)_p(\d+)_f(\d+)\.jpg", fp.name)
            if m:
                rows.append(dict(video_id=video_id, video_path=str(video_path),
                                 frame_path=fp.relative_to(OUT_DIR).as_posix(),
                                 frame_index=int(m.group(3)), time_sec="",
                                 source=source, stem=video_path.stem, orig_w="", orig_h="",
                                 window_index=int(m.group(1)), pos_in_window=int(m.group(2))))
        n_ok[source] += 1
        n_by_root[f"{source} <- {root.name}"] += 1
        continue

    try:
        new_rows = extract_one(video_path, video_id, out_dir, source)
    except DurationSkip:
        n_duration[source] += 1
        continue
    except Exception as e:
        n_corrupt[source] += 1
        shutil.rmtree(out_dir, ignore_errors=True)
        n_logged += 1
        if n_logged <= 20:
            print("SKIP:", video_path.name, "-", str(e).splitlines()[0][:80])
        continue

    rows.extend(new_rows)
    n_ok[source] += 1
    n_by_root[f"{source} <- {root.name}"] += 1
    pbar.set_postfix({f"{k}_ok": v for k, v in n_ok.items()})

    if all(n_ok[k] >= targets[k] for k in targets):
        break

with open(INDEX_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["video_id", "video_path", "frame_path",
                                           "frame_index", "time_sec", "source", "stem",
                                           "orig_w", "orig_h", "window_index", "pos_in_window"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\n=== extraction summary (this run) ===")
for label in targets:
    print(f"{label:14s} clean={n_ok[label]:4d}/{targets[label]:<4d} "
          f"corrupt={n_corrupt[label]:3d}  bad-duration={n_duration[label]:3d}")
if n_prev_skipped:
    print(f"resumed        : skipped {n_prev_skipped} videos already extracted by a previous run")
print("\nper dataset (AUDIT: the NEW batches MUST show non-zero here):")
for key, n in sorted(n_by_root.items()):
    print(f"  {key:45s} {n:5d} videos")
print(f"total frames    : {len(rows)}")
print(f"index -> {INDEX_CSV}  (frame_path RELATIVE to {OUT_DIR})")
if budget_hit:
    print("\n*** PARTIAL RUN (time budget). To finish: run the packaging cell below,")
    print("    Save Version, publish THIS output as a dataset, attach it to a fresh run")
    print("    of this notebook, and re-Commit -- the resume cell skips everything done.")
    print("    The training notebook merges ALL extraction tars automatically.")
for label in targets:
    if n_ok[label] < targets[label] and not budget_hit:
        print(f"\n*** {label} SHORT by {targets[label] - n_ok[label]}: ran out of candidates.")
        print(f"    Add more {label} source videos to SOURCES and re-run -- dedupe skips")
        print(f"    ones already used, and existing extracted frames are reused, not redone.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(INDEX_CSV)
print(df.groupby("source")["video_id"].nunique().rename("videos"))
print(df.groupby("source").size().rename("frames"))

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax_row, label in zip(axes, ["real", "ai_generated"]):
    pool = df[df.source == label].drop_duplicates("video_id")
    sample = pool.sample(min(6, len(pool)), random_state=0)
    for ax, (_, row) in zip(ax_row, sample.iterrows()):
        img = cv2.cvtColor(cv2.imread(str(OUT_DIR / row.frame_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{label}: {row.video_id[:18]}", fontsize=7)
    for ax in ax_row:
        ax.axis("off")
plt.tight_layout(); plt.show()

one = df[(df.video_id == df.video_id.iloc[0]) & (df.window_index == 0)].sort_values("pos_in_window")
k = min(8, len(one))
fig, axes = plt.subplots(1, k, figsize=(2 * k, 2.4))
for ax, (_, row) in zip(np.atleast_1d(axes), one.head(k).iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(str(OUT_DIR / row.frame_path)), cv2.COLOR_BGR2RGB))
    ax.set_title(f"pos {int(row.pos_in_window)}\nframe {int(row.frame_index)}", fontsize=7)
    ax.axis("off")
fig.suptitle(f"one contiguous window ({one.video_id.iloc[0][:24]}) -- consecutive frames = motion", fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
import shutil, subprocess

tar_path = f"/kaggle/working/chronos_frames_{BATCH_TAG}_{RUN_STAMP}.tar"
subprocess.run(["tar", "-cf", tar_path, "-C", str(OUT_DIR),
                "frames", INDEX_CSV.name], check=True)
shutil.copy(INDEX_CSV, "/kaggle/working/")
shutil.rmtree(FRAMES_DIR)
gb = Path(tar_path).stat().st_size / 1e9
print(f"{tar_path} ready ({gb:.2f} GB)")
assert gb < 19.0, ("tar too close to Kaggle's ~19.5GB output cap -- lower "
                   "TARGET_PER_CLASS or JPEG_QUALITY and re-run")


## Next steps
1. Smoke run looked right (dedupe report sane, preview grid shows real
   frames that actually look different from AI-generated ones, per-class
   summary has few corrupt/bad-duration)? Set `LIMIT_PER_CLASS = None` →
   **Save Version → Save & Run All (Commit)**.
2. Output tab will hold `chronos_frames_combined.tar` +
   `frames_index_combined.csv` -- containing BOTH labels already, no
   merge step required.
3. **Manifest + training notebook** (next stage): attach this notebook's
   output via *Add Input → Your Work*, untar into `data/`, drop the CSV
   next to it, then run `build_manifest.py` (it globs
   `frames_index*.csv` automatically) followed by `train.py` -- which now
   has both classes and can actually train.

**If a class finishes short:** the summary tells you by how much. Attach
one more dataset for that label, add it to `SOURCES`, and re-run --
already-extracted videos are detected and skipped (idempotent), so you
only pay for the shortfall.

**Consistency rule for later:** v1 trains on full resize-to-square frames, so
inference preprocesses the same way (resize-square, NOT face-crop) --
already wired up in `training/src/inference.py`.